# Table 2 — outcomes by arm, observed and weighted

**What happened to whom, before any estimate.** Table 1 describes who was compared; this describes
what they experienced. Every outcome the study reports — the seven 90-day mRS classes, the three
secondary outcomes and the four safety outcomes — as counts in the arms as observed, and as shares in
the overlap-weighted population the primary estimate is about.

It carries no effect, no interval and no p value. Those are the supplement. What this table is for is
the thing an effect estimate cannot show: how many patients were in each state.

This notebook **composes**; it computes nothing. Section references `[§n]` point to
`../statistical_analysis_plan.md`.

## 1. The run

One cached build of Stages 1-11, shared by all the exhibits.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Independent of where the notebook was started: find the directory the shipped modules live in.
_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.py").exists())
sys.path[:0] = [str(_root), str(_root / "figures_and_tables")]

import config as C
import report
import manuscript as M

M.use_house_style()
ARM = C.TREATMENT_LABELS          # {0: "EVT alone", 1: "bridging"}
ARMS = sorted(ARM)

bundle = M.load()                 # the cached Stages 1-11 run; built once, then read in seconds
bundle.provenance

## 2. The rows

`report.outcome_summary` walks the mRS classes and then each family of binary outcomes. The observed
columns are counts in each arm; the weighted columns come off the estimates themselves — the mRS
classes from the same weighted distribution Figure 3 plots, each binary outcome from its own weighted
proportion — so this table cannot disagree with them.

Each outcome carries its own denominator `[§11]`: one of them is recorded on fewer patients than the
rest, and the `n` column is where that shows.

In [ ]:
rows = report.outcome_summary(bundle)
table = pd.DataFrame(list(rows[1:]), columns=list(rows[0]))
table

### The weighted mRS classes are Figure 3's panel A

Worth checking rather than asserting: the weighted shares here and the distribution the primary
figure draws are one set of numbers.

In [ ]:
plotted = {ARM[a]: report.pmf(bundle.primary.cumulative, a) for a in ARMS}
pd.DataFrame(plotted, index=[f"mRS {j}" for j in C.MRS_LEVELS]).T.round(3)

## 3. Write it

In [ ]:
path = M.save_table("Table_2", table, (
    "Table 2. Outcomes by arm in the [§7] cohort: counts as observed, and shares in the "
    "overlap-weighted population. The mRS classes are the distribution summarised by the common odds "
    "ratio in Table S1; the binary outcomes are those estimated in Table S2. No effect estimate, "
    "interval or p value appears here.",
    report.DESCRIPTIVE_CRUDE,
    report.ATO_DESCRIPTION,
    "Every estimate carries its own denominator [§11]; the observed and weighted columns of a row "
    "describe the same patients.",
    M.provenance_clause(bundle)))

print(path)
pd.read_csv(path, comment="#").head(9)

## 4. Reading it

- **The observed columns are not an unadjusted effect** `[§16b]`. Treatment is nearly determined by
  centre, so a pooled crude comparison can differ from the within-centre one in *direction* — the
  caption says so, and Figure 2 panel B is why.
- **The weighted columns describe the overlap population**, not the cohort `[§7]`: patients whose
  treatment was genuinely in equipoise. They are what Table S1's estimate is about.
- **Read the mRS classes as a distribution, not as seven comparisons.** The one number summarising
  them is the common odds ratio, and its caveat travels with it in Table S1.